In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from aquarel import load_theme

theme = load_theme("minimal_light")
theme.apply()
mpl.rcParams['axes.spines.left'] = True
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.bottom'] = True
mpl.rcParams['axes.edgecolor'] = 'grey'

In [ ]:
FIGURE_DATA = Path("../figure_data")

ES_INPUT_MAP = {'es0': 3.0, 'es1': 2.0, 'es2': 1.0, 'es3': 0.75, 'es4': 0.5, 'es5': 0.25}

METHOD_RENAME = {
    'HTreWAS Terminal':     'TreeWAS',
    'HTreWAS Subsequent':   'TreeWAS Subsequent',
    'HTreWAS Simultaneous': 'TreeWAS Simultaneous',
}

df_all = pd.read_csv(FIGURE_DATA / 'all_metrics.csv')
df_all['Method'] = df_all['Method'].replace(METHOD_RENAME)
df_all['Power']  = df_all['effect_size'].map(ES_INPUT_MAP)

# Aggregate across trees (per Method, Power)
# Exclude groupby keys from numeric_cols to avoid reset_index collision
numeric_cols = [c for c in df_all.select_dtypes(include='number').columns
                if c not in ['Power']]
df     = df_all.groupby(['Method', 'Power'])[numeric_cols].mean().reset_index()
df_std = df_all.groupby(['Method', 'Power'])[numeric_cols].std().reset_index()

In [ ]:
order = ['SimPhyNI', 'FaST-LMM', 'TreeWAS', 'Pagel', 'Scoary', 'Coinfinder', 'SpydrPick', "Fisher's"]

In [ ]:
def plot_metric_with_errorbars(
    df,
    df_std,
    metric,
    exclude_methods=None,
    log_scale=False,
    method_order=None,
    xlim=None,
):
    if exclude_methods is None:
        exclude_methods = []

    df = df.copy()

    # Enforce strict method ordering if provided
    if method_order:
        df = df[df['Method'].isin(method_order)]
        df['Method'] = pd.Categorical(df['Method'], categories=method_order, ordered=True)
        df = df.sort_values(['Method', 'Power'])
        df_std = df_std[df_std['Method'].isin(method_order)]
        df_std['Method'] = pd.Categorical(df_std['Method'], categories=method_order, ordered=True)
        df_std = df_std.sort_values(['Method', 'Power'])
    else:
        # Default: SimPhyNI first
        df['SortOrder'] = df['Method'].apply(lambda x: (0, x) if x == 'SimPhyNI' else (1, x))
        df = df.sort_values(by=['SortOrder', 'Method', 'Power']).drop(columns='SortOrder')
        df_std = df_std[df_std['Method'].isin(df['Method'])]

    fig, ax = plt.subplots(figsize=(4, 3))
    handles, labels = [], []

    for method in df['Method'].unique():
        if method in exclude_methods:
            continue

        subset = df[df['Method'] == method]
        std_subset = df_std[df_std['Method'] == method]

        y_values = subset[metric].astype(float)
        y_errors = std_subset[metric].astype(float).values if metric in std_subset.columns else None

        is_focus = (method == "SimPhyNI")
        lw = 2.8 if is_focus else 2.0
        ms = 8 if is_focus else 6.5
        a  = .9 if is_focus else 0.75
        zo = 10 if is_focus else 4

        line = ax.errorbar(
            subset['Power'],
            y_values,
            yerr=y_errors,
            label=method,
            capsize=4,
            capthick=2.0,
            marker='o',
            alpha=a,
            zorder=zo
        )

        handles.append(line)
        labels.append(method)

    if log_scale:
        ax.set_yscale('log')
    if xlim:
        ax.set_xlim(*xlim)
    ax.set_xlabel('Interaction Strength')
    ax.set_ylabel(metric.replace('_', ' '))
    plt.tight_layout()

    plt.savefig('fig.svg', format='svg')

    # Plot legend in a new figure
    fig_leg, ax_leg = plt.subplots(figsize=(3, 4))
    ax_leg.axis('off')
    ax_leg.legend(handles, labels, loc='center', title='Method')
    plt.tight_layout()
    plt.show()


def plot_metric_bar(
    df,
    metric,
    orientation="vertical",
    power=2,
    exclude_methods=None,
    color_map=plt.cm.tab10.colors,
    title=None,
    ylim=(0, 1.15),
    method_order=None,
    log=False,
):
    df_filt = df[df['Power'] == power]
    if exclude_methods:
        df_filt = df_filt[~df_filt['Method'].isin(exclude_methods)]

    if method_order:
        df_filt = df_filt[df_filt['Method'].isin(method_order)]
        df_filt['Method'] = pd.Categorical(df_filt['Method'], categories=method_order, ordered=True)
        df_filt = df_filt.sort_values('Method')

    colors = color_map[:len(df_filt)]

    plt.figure(figsize=(4, 4))
    if orientation == "vertical":
        bars = plt.bar(df_filt['Method'], df_filt[metric].astype(float), color=colors)
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(metric)
        if log: plt.yscale('log')
        plt.ylim(*ylim)
        plt.bar_label(bars, padding=3, fmt='%.3f')
    else:
        bars = plt.barh(df_filt['Method'], df_filt[metric].astype(float), color=colors)
        plt.xlabel(metric)
        if log: plt.xscale('log')
        plt.xlim(*ylim)
        plt.bar_label(bars, padding=3, fmt='%.3f')

    if title:
        plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_metric_box(
    df_all,
    metric,
    power,
    exclude_methods=None,
    title=None,
    ylim=(-0.15, 1.15),
    method_order=None,
    log=False,
    baseline=None,
    label_stat="mean",  # or "median"
):
    if type(power) == list:
        df_filt = df_all[df_all['Power'].isin(power)]
    else:
        df_filt = df_all[df_all['Power'] == power]
    if exclude_methods:
        df_filt = df_filt[~df_filt['Method'].isin(exclude_methods)]

    if method_order:
        df_filt = df_filt[df_filt['Method'].isin(method_order)]
        df_filt['Method'] = pd.Categorical(df_filt['Method'], categories=method_order, ordered=True)
        df_filt = df_filt.sort_values('Method')

    plt.figure(figsize=(3, 3))
    ax = plt.gca()

    # Strip and box plots
    sns.stripplot(data=df_filt, x='Method', y=metric, color='black', alpha=0.6, jitter=True, dodge=True, size=4, ax=ax)
    sns.boxplot(data=df_filt, x='Method', y=metric, hue='Method', palette='tab10', showfliers=False, linewidth=0.5, ax=ax)

    # Optional baseline
    if baseline is not None:
        ax.axhline(baseline, linestyle='--', color='gray', label=f'baseline {metric}')

    # Add text labels for each method
    grouped = df_filt.groupby('Method')[metric]
    for i, method in enumerate(df_filt['Method'].cat.categories):
        if label_stat == "mean":
            val = grouped.mean().get(method, np.nan)
        else:
            val = grouped.median().get(method, np.nan)
        if not np.isnan(val):
            ax.text(i, max(df_filt[df_filt['Method'] == method][metric]) + 0.005, f"{val:.3f}", ha='center', va='bottom', fontsize=8)

    # Labels and formatting
    plt.xticks(rotation=45, ha='right')
    plt.xlabel("")
    plt.ylabel(metric.replace('_', ' '))
    if log: plt.yscale('log')
    plt.ylim(*ylim)
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.savefig('fig.svg', format='svg')
    plt.show()

    # === Statistical comparison (first column vs others) ===
    methods = df_filt['Method'].cat.categories
    baseline_method = methods[0]
    baseline_vals = df_filt[df_filt['Method'] == baseline_method][metric]

    report = []
    for method in methods[1:]:
        vals = df_filt[df_filt['Method'] == method][metric]
        if len(vals) > 0:
            # Mann-Whitney U test (non-parametric, independent samples)
            stat, pval = mannwhitneyu(baseline_vals, vals, alternative="two-sided")
            report.append({
                "baseline": baseline_method,
                "method": method,
                "statistic": stat,
                "p_value": pval,
                "n_baseline": len(baseline_vals),
                "n_method": len(vals),
                "mean_baseline": baseline_vals.mean(),
                "mean_method": vals.mean(),
            })

    report_df = pd.DataFrame(report)
    return report_df


def plot_all_metrics_bars(
    df,
    power,
    exclude_methods=None,
    metrics=None,
    color_map=plt.cm.tab10.colors,
    method_order=None
):
    if metrics is None:
        metrics = [
            'Precision_Positive', 'Recall_Positive', 'F1_Positive', 'AUC_ROC_Positive', 'PR_AUC_Positive', 'FPR_Positive',
            'Precision_Negative', 'Recall_Negative', 'F1_Negative', 'AUC_ROC_Negative', 'PR_AUC_Negative', 'FPR_Negative'
        ]

    df_filt = df[df['Power'] == power]
    if exclude_methods:
        df_filt = df_filt[~df_filt['Method'].isin(exclude_methods)]

    if method_order:
        df_filt = df_filt[df_filt['Method'].isin(method_order)]
        df_filt['Method'] = pd.Categorical(df_filt['Method'], categories=method_order, ordered=True)
        df_filt = df_filt.sort_values('Method')

    plt.figure(figsize=(20, 16))
    for i, metric in enumerate(metrics, start=1):
        plt.subplot(4, 3, i)
        bars = plt.bar(df_filt['Method'], df_filt[metric].astype(float), color=color_map)  # type: ignore
        plt.title(metric)
        plt.ylabel(metric)
        plt.ylim(0, 1.15)
        plt.xticks(rotation=45, ha='right')
        plt.bar_label(bars, padding=3, fmt='%.3f')

    plt.tight_layout()
    plt.show()


def plot_all_metrics_bars_std(
    df,
    df_std,
    power=2,
    exclude_methods=None,
    metrics=None,
    color_map=plt.cm.tab10.colors,
    method_order=None
):
    if metrics is None:
        metrics = [
            'Precision_Positive', 'Recall_Positive', 'F1_Positive', 'AUC_ROC_Positive', 'PR_AUC_Positive', 'FPR_Positive',
            'Precision_Negative', 'Recall_Negative', 'F1_Negative', 'AUC_ROC_Negative', 'PR_AUC_Negative', 'FPR_Negative'
        ]

    df_filt = df[df['Power'] == power]
    df_std_filt = df_std[df_std['Power'] == power]

    if exclude_methods:
        df_filt = df_filt[~df_filt['Method'].isin(exclude_methods)]
        df_std_filt = df_std_filt[~df_std_filt['Method'].isin(exclude_methods)]

    if method_order:
        df_filt = df_filt[df_filt['Method'].isin(method_order)]
        df_std_filt = df_std_filt[df_std_filt['Method'].isin(method_order)]
        df_filt['Method'] = pd.Categorical(df_filt['Method'], categories=method_order, ordered=True)
        df_std_filt['Method'] = pd.Categorical(df_std_filt['Method'], categories=method_order, ordered=True)
        df_filt = df_filt.sort_values('Method')
        df_std_filt = df_std_filt.sort_values('Method')

    plt.figure(figsize=(20, 16))
    for i, metric in enumerate(metrics, start=1):
        plt.subplot(4, 3, i)

        yerr = df_std_filt[metric].astype(float).values  # Standard deviation as error bars

        bars = plt.bar(
            df_filt['Method'],
            df_filt[metric].astype(float),
            color=color_map,
            yerr=yerr,
            capsize=5,
        )

        plt.title(metric)
        plt.ylabel(metric)
        plt.ylim(0, 1.15)
        plt.xticks([])
        plt.bar_label(bars, padding=3, fmt='%.3f')
    plt.legend(bars, df_filt['Method'], bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.savefig('fig.svg', format='svg')
    plt.show()


def plot_all_metrics_box(
    df_all,
    power=2,
    exclude_methods=None,
    metrics=None,
    color_map=plt.cm.tab10.colors,
    method_order=None
):
    if metrics is None:
        metrics = [
            'Precision_Positive', 'Recall_Positive', 'F1_Positive', 'AUC_ROC_Positive', 'PR_AUC_Positive', 'FPR_Positive',
            'Precision_Negative', 'Recall_Negative', 'F1_Negative', 'AUC_ROC_Negative', 'PR_AUC_Negative', 'FPR_Negative'
        ]

    df_filt = df_all[df_all['Power'] == power]
    if exclude_methods:
        df_filt = df_filt[~df_filt['Method'].isin(exclude_methods)]
    if method_order:
        df_filt = df_filt[df_filt['Method'].isin(method_order)]
        df_filt['Method'] = pd.Categorical(df_filt['Method'], categories=method_order, ordered=True)
        df_filt = df_filt.sort_values('Method')
    plt.figure(figsize=(20, 16))
    for i, metric in enumerate(metrics, start=1):
        ax = plt.subplot(4, 3, i)
        sns.stripplot(data=df_filt, x='Method', y=metric, color='black', alpha=0.6, jitter=True, dodge=True, size=4, ax=ax)
        sns.boxplot(data=df_filt, x='Method', y=metric, palette='tab10', showfliers=False, linewidth=0.5, ax=ax)

        grouped = df_filt.groupby('Method')[metric]
        for i, method in enumerate(df_filt['Method'].cat.categories):
            val = grouped.median().get(method, np.nan)
            if not np.isnan(val):
                ax.text(i, max(df_filt[df_filt['Method'] == method][metric]) + 0.005, f"{val:.3f}", ha='center', va='bottom', fontsize=8)

        plt.title(metric)
        plt.ylabel(metric)
        plt.ylim(-0.1, 1.15)
        plt.xticks([])
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.savefig('fig.svg', format='svg')
    plt.show()

In [ ]:
### Figure 3C (1/2)
plot_metric_box(df_all, 'FPR_Positive', power=0.75, method_order=order, ylim=(-0.01, .15))

In [ ]:
### Figure 3C (2/2)
plot_metric_box(df_all, 'FPR_Negative', power=[0.75], method_order=order, ylim=(-0.01, .15))

In [ ]:
### Figure 3B (1/2)
plot_metric_box(df_all, 'PR_AUC_Positive', power=[0.75], method_order=order, ylim=(0, 1.05), baseline=1/11)

In [ ]:
### Figure 3B (2/2)
plot_metric_box(df_all, 'PR_AUC_Negative', power=[0.75], method_order=order, ylim=(0, 1.05), baseline=1/11)

In [ ]:
### Figure S8
plot_all_metrics_bars_std(df, df_std, power=0.75, method_order=order, exclude_methods=[])

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='PR_AUC_Positive', log_scale=False, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='PR_AUC_Negative', log_scale=False, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='FPR_Positive', log_scale=True, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='FPR_Negative', log_scale=True, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='Precision_Positive', log_scale=False, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='Precision_Negative', log_scale=False, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='Recall_Positive', log_scale=False, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='Recall_Negative', log_scale=False, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='F1_Positive', log_scale=False, method_order=order)

In [ ]:
plot_metric_with_errorbars(df, df_std, metric='F1_Negative', log_scale=False, method_order=order)